# 05 — Explicabilidad y evaluación más allá del accuracy

- **Permutación:** rompe una feature en test y mide caída de rendimiento.
- **SHAP** (árboles): opcional con `pip install -e ".[explain]"`.

**Siguiente:** `06_pipeline_Kedro.ipynb`.

In [ ]:
%load_ext kedro.ipython

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "raw" / "database.sqlite"
assert (PROJECT_ROOT / "pyproject.toml").is_file(), (
    "Abre este notebook desde la raíz del proyecto con `kedro jupyter lab`."
)
assert DB_PATH.is_file(), (
    "No existe data/raw/database.sqlite. Ejecuta: `python scripts/bootstrap_data.py`."
)
print(f"Proyecto: {PROJECT_ROOT}")
print(f"SQLite: {DB_PATH}")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

match = catalog.load("match")
goal_diff = match["home_team_goal"] - match["away_team_goal"]
outcome = np.select(
    [goal_diff > 0, goal_diff == 0, goal_diff < 0],
    [2, 1, 0],
)
feat_cols = ["B365H", "B365D", "B365A"]
df = match[feat_cols].assign(outcome=outcome, home_team_goal=match["home_team_goal"]).dropna()
X = df[feat_cols]
y_clf = df["outcome"]
y_reg = df["home_team_goal"]

# Un solo split: mismas filas en train/test para clasificación y regresión
idx = np.arange(len(X))
idx_train, idx_test = train_test_split(
    idx, test_size=0.2, random_state=42, stratify=y_clf
)
X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train_c, y_test_c = y_clf.iloc[idx_train], y_clf.iloc[idx_test]
y_train_r, y_test_r = y_reg.iloc[idx_train], y_reg.iloc[idx_test]

clf = RandomForestClassifier(
    n_estimators=200, max_depth=16, min_samples_leaf=5, random_state=42, n_jobs=1
)
clf.fit(X_train, y_train_c)
reg = RandomForestRegressor(
    n_estimators=200, max_depth=16, min_samples_leaf=5, random_state=42, n_jobs=1
)
reg.fit(X_train, y_train_r)

perm_c = permutation_importance(
    clf, X_test, y_test_c, n_repeats=15, random_state=42, n_jobs=1
)
perm_r = permutation_importance(
    reg, X_test, y_test_r, n_repeats=15, random_state=42, n_jobs=1
)

imp_c = pd.DataFrame(
    {"feature": feat_cols, "mean": perm_c.importances_mean, "std": perm_c.importances_std}
).sort_values("mean", ascending=False)
imp_r = pd.DataFrame(
    {"feature": feat_cols, "mean": perm_r.importances_mean, "std": perm_r.importances_std}
).sort_values("mean", ascending=False)

print("Clasificación (RF) — importancia por permutación")
display(imp_c)
print("Regresión (RF) — importancia por permutación")
display(imp_r)

In [ ]:
# Coeficientes de Logística (multiclase: un vector por clase en coef_)
log_pipe = Pipeline(
    [
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2_000, solver="lbfgs")),
    ]
)
log_pipe.fit(X_train, y_train_c)
coef = log_pipe.named_steps["clf"].coef_
feat = log_pipe.named_steps["scale"].get_feature_names_out(feat_cols)
pd.DataFrame(coef, columns=feat, index=["class_0_away", "class_1_draw", "class_2_home"])

In [ ]:
try:
    import shap
except ImportError:
    print('Instala SHAP: pip install -e ".[explain]"')
else:
    sample = X_test.sample(min(400, len(X_test)), random_state=42)
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(sample)
    vals = sv[2] if isinstance(sv, list) else sv
    shap.summary_plot(vals, sample, feature_names=feat_cols, show=True)